In [1]:
import duckdb

In [2]:
duckdb.sql("select 1 as test").df()

,test
0,1


In [3]:
import pandas as pd

sheets = pd.read_excel("market_data.xlsx", sheet_name=None)
for name, d in sheets.items():
    print(f"Sheet: {name}, Shape: {d.shape}")

Sheet: source_A_industry_assoc, Shape: (37, 5)
Sheet: source_B_research_firm, Shape: (47, 5)
Sheet: drivers, Shape: (48, 4)
Sheet: fx_rates, Shape: (3, 2)


In [4]:
a = sheets["source_A_industry_assoc"]
b = sheets["source_B_research_firm"]
drivers = sheets["drivers"]
fx = sheets["fx_rates"]

In [5]:
duckdb.sql("select * from a limit 10").df()

,country,year,market_value,currency,unit
0,Germany,2018,18.2,EUR,million
1,Germany,2019,19.7,EUR,million
2,Germany,2020,18.1,EUR,million
3,Germany,2021,23.6,EUR,million
4,Germany,2022,25.6,EUR,million
5,Germany,2023,28.6,EUR,million
6,Germany,2024,29.9,EUR,million
7,Germany,2025,34.6,EUR,million
8,France,2018,12.6,EUR,million
9,France,2019,13.3,EUR,million


In [6]:
duckdb.sql("""
select currency, unit, count(*) as n
from a
group by currency, unit
""").df()

,currency,unit,n
0,EUR,million,37


In [7]:
duckdb.sql("""
select country, currency, unit, count(*) as n
from b
group by country, currency, unit
order by country""").df()

,country,currency,unit,n
0,Austria,EUR,billion,8
1,Denmark,DKK,million,8
2,France,EUR,billion,8
3,Germany,EUR,billion,7
4,Netherlands,EUR,billion,8
5,Poland,PLN,million,8


In [8]:
duckdb.sql("""
select country,
count(year) as n,
min(year) as min_year,
max(year) as max_year
from a
group by country
order by country""").df()

,country,n,min_year,max_year
0,Austria,7,2018,2025
1,France,8,2018,2025
2,Germany,8,2018,2025
3,Netherlands,6,2018,2025
4,Poland,8,2018,2025


In [10]:
duckdb.sql("""
select * from a
where country = 'France'
order by year""").df()

,country,year,market_value,currency,unit
0,France,2018,12.6,EUR,million
1,France,2019,13.3,EUR,million
2,France,2020,12.5,EUR,million
3,France,2021,16.3,EUR,million
4,France,2022,176.0,EUR,million
5,France,2023,20.2,EUR,million
6,France,2024,21.4,EUR,million
7,France,2025,23.6,EUR,million


Standardise units and currencies

In [12]:
duckdb.sql("select * from fx").df()

,currency,eur_per_unit
0,EUR,1.000000
1,PLN,0.232558
2,DKK,0.134228


In [20]:
a_std_sql = duckdb.sql("""
    SELECT
        a.country,
        a.year,
        a.market_value,
        a.currency,
        a.unit,
        a.market_value
          * CASE a.unit WHEN 'million' THEN 1 WHEN 'billion' THEN 1000 END
          * fx.eur_per_unit AS value_eur_m
    FROM a
    LEFT JOIN fx ON a.currency = fx.currency
    ORDER BY a.country, a.year
""").df()
a_std_sql.head(10)

,country,year,market_value,currency,unit,value_eur_m
0,Austria,2018,2.0,EUR,million,2.0
1,Austria,2019,2.2,EUR,million,2.2
2,Austria,2020,1.9,EUR,million,1.9
3,Austria,2021,2.5,EUR,million,2.5
4,Austria,2022,2.6,EUR,million,2.6
5,Austria,2024,3.0,EUR,million,3.0
6,Austria,2025,3.3,EUR,million,3.3
7,France,2018,12.6,EUR,million,12.6
8,France,2019,13.3,EUR,million,13.3
9,France,2020,12.5,EUR,million,12.5


In [21]:
b_std_sql = duckdb.sql("""
    SELECT
        b.country,
        b.year,
        b.market_value,
        b.currency,
        b.unit,
        b.market_value
          * CASE b.unit WHEN 'million' THEN 1 WHEN 'billion' THEN 1000 END
          * fx.eur_per_unit AS value_eur_m
    FROM b
    LEFT JOIN fx ON b.currency = fx.currency
    ORDER BY b.country, b.year
""").df()
b_std_sql.head(10)

,country,year,market_value,currency,unit,value_eur_m
0,Austria,2018,0.002,EUR,billion,2.000000
1,Austria,2019,0.002,EUR,billion,2.000000
2,Austria,2020,0.002,EUR,billion,2.000000
3,Austria,2021,0.003,EUR,billion,3.000000
4,Austria,2022,0.003,EUR,billion,3.000000
5,Austria,2023,0.003,EUR,billion,3.000000
6,Austria,2024,0.003,EUR,billion,3.000000
7,Austria,2025,0.004,EUR,billion,4.000000
8,Denmark,2018,14.000,DKK,million,1.879195
9,Denmark,2019,15.000,DKK,million,2.013423


In [22]:
duckdb.sql("""
    SELECT COUNT(*) AS rows_without_fx
    FROM b
    LEFT JOIN fx ON b.currency = fx.currency
    WHERE fx.eur_per_unit IS NULL
""").df()

,rows_without_fx
0,0


In [23]:
duckdb.sql("""
    SELECT a.country, a.year,
           a.value_eur_m AS value_a,
           b.value_eur_m AS value_b,
           ROUND(b.value_eur_m / a.value_eur_m, 2) AS ratio_b_to_a
    FROM a_std AS a
    INNER JOIN b_std AS b
        ON a.country = b.country AND a.year = b.year
    ORDER BY a.country, a.year
""").df()

CatalogException: Catalog Error: Table with name a_std does not exist!
Did you mean "pg_attrdef"?

In [24]:
#quality check
duckdb.sql("""
    with countries as (
        select distinct country from a_std
        union
        select disticnt country from b_std),
        years as (
            select * from range(2018,2026) as t)
        select c.country, y.year
        from countries c
        cross join years y
        order by c.country, y.year""")

CatalogException: Catalog Error: Table with name a_std does not exist!
Did you mean "pg_attrdef"?